# Day 11 · 딥러닝 실습 — 신경망에서 VLM까지

Colab 무료 GPU(T4)에서 PyTorch로 진행한다. 슬라이드의 흐름을 코드로 직접 돌려 본다.

> **시작 전 필수** — 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU → 저장.** Lab 4의 CPU/GPU 비교와 전이학습에 GPU가 필요하다.

| 절 | 내용 | 확인 |
|---|---|---|
| Lab 0 | 환경·GPU 확인 | torch·CUDA |
| Lab 1 | MNIST — 신경망(MLP) 첫 훈련 | 학습 루프·정확도 |
| Lab 2 | CIFAR-10 — MLP의 한계 → CNN | 정확도 비교 |
| Lab 3 | 전이학습 — ResNet의 FC만 교체 | 소량으로 높은 정확도 |
| Lab 4 | CPU vs GPU — 같은 학습 시간 비교 | 속도 체감 |
| Lab 5 | VLM — 이미지를 말로 묻기 (NVIDIA API) | 멀티모달 |

PyTorch·torchvision은 Colab에 이미 설치돼 있다. Lab 5만 `openai` 패키지를 추가로 받는다.

## Lab 0 · 환경·GPU 확인

`cuda`가 나오면 GPU 사용 준비 완료다. `cpu`가 나오면 런타임 유형이 CPU다 — 런타임 유형 변경 후 다시 실행한다.

In [ ]:
!nvidia-smi -L
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "· device =", device)

## Lab 1 · MNIST — 첫 신경망(MLP)

손글씨 숫자 28×28을 한 줄(784칸)로 펴서 넣는다. `nn.Module`로 층을 쌓고, 학습 루프(판단→손실→역전파→갱신)를 돈다.

### 먼저 · NumPy 배열과 텐서

딥러닝의 데이터는 전부 **텐서**다. 텐서는 사실상 NumPy 배열과 같고, 두 가지가 더 있다 — GPU에서 돌고, 어떻게 계산됐는지 기억해 기울기를 구한다.

In [ ]:
import numpy as np, torch

a = np.array([[1., 2.], [3., 4.]])       # NumPy 배열 (CPU)
t = torch.tensor([[1., 2.], [3., 4.]])   # 텐서 — 모양·연산이 같다

print("shape  ", a.shape, "==", tuple(t.shape))
print("행렬 곱 numpy\n", a @ a)
print("행렬 곱 torch\n", (t @ t).numpy())

# 서로 오갈 수 있다
print("배열→텐서", torch.from_numpy(a).dtype)
print("텐서→배열", t.numpy().dtype)

# 텐서에만 있는 것: GPU로 옮기기, 기울기 추적
t_gpu = t.to(device)                     # ← NumPy엔 없는 것
g = torch.tensor([2.0], requires_grad=True)   # 기울기를 기억
(g**2).backward(); print("d(g^2)/dg =", g.grad.item(), "(= 2g = 4)")

In [ ]:
import torch, torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

tf = transforms.ToTensor()
train = datasets.MNIST("data", train=True,  download=True, transform=tf)
test  = datasets.MNIST("data", train=False, download=True, transform=tf)
train_dl = DataLoader(train, batch_size=128, shuffle=True)
test_dl  = DataLoader(test,  batch_size=256)
print("학습", len(train), "· 테스트", len(test), "· 이미지 shape", train[0][0].shape)

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),            # 28x28 -> 784
            nn.Linear(784, 128), nn.ReLU(),   # 은닉층 + 활성화
            nn.Linear(128, 10),      # 10개 클래스 점수
        )
    def forward(self, x):
        return self.net(x)

def run_epoch(model, dl, opt=None):
    train = opt is not None
    model.train(train)
    loss_fn = nn.CrossEntropyLoss()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        with torch.set_grad_enabled(train):
            out = model(x)
            loss = loss_fn(out, y)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()   # 학습 루프의 세 줄
        loss_sum += loss.item()*len(y)
        correct += (out.argmax(1) == y).sum().item(); total += len(y)
    return loss_sum/total, correct/total

In [ ]:
model = MLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(3):
    tr_loss, tr_acc = run_epoch(model, train_dl, opt)
    te_loss, te_acc = run_epoch(model, test_dl)
    print(f"epoch {epoch+1}: train acc {tr_acc:.3f} · test acc {te_acc:.3f}")

### 관찰
- 3에폭이면 테스트 정확도 약 0.97. 손글씨 숫자는 MLP만으로도 잘 맞힌다.
- `zero_grad → backward → step` 세 줄이 학습의 전부다 — 기울기를 0으로 비우고, 역전파로 방향을 구하고, 그 방향으로 한 걸음 간다.

## Lab 2 · CIFAR-10 — MLP의 한계, 그리고 CNN

컬러 사진 32×32×3(3,072칸)은 MNIST와 다르다. **펴서 넣으면 이웃한 픽셀 관계가 사라진다.** 먼저 MLP로 해보고 한계를 본 뒤, 합성곱(CNN)을 얹는다.

In [ ]:
from torchvision import datasets, transforms
tf = transforms.ToTensor()
ctrain = datasets.CIFAR10("data", train=True,  download=True, transform=tf)
ctest  = datasets.CIFAR10("data", train=False, download=True, transform=tf)
ctrain_dl = DataLoader(ctrain, batch_size=128, shuffle=True)
ctest_dl  = DataLoader(ctest,  batch_size=256)
classes = ctrain.classes
print("클래스", classes, "· 이미지 shape", ctrain[0][0].shape)

In [ ]:
# (1) MLP를 CIFAR-10에 — 펴서 넣는 방식
class MLP_C(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                    # 3x32x32 -> 3072
            nn.Linear(3072, 256), nn.ReLU(),
            nn.Linear(256, 10))
    def forward(self, x): return self.net(x)

mlp_c = MLP_C().to(device)
opt = torch.optim.Adam(mlp_c.parameters(), lr=1e-3)
for epoch in range(3):
    _, tr = run_epoch(mlp_c, ctrain_dl, opt)
    _, te = run_epoch(mlp_c, ctest_dl)
print(f"[MLP] CIFAR-10 test acc {te:.3f}   ← 사진에서는 잘 안 오른다")

In [ ]:
# (2) CNN — 합성곱으로 무늬를 보고, 풀링으로 줄인다
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32x16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 64x8x8
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(64*8*8, 128), nn.ReLU(), nn.Linear(128, 10))
    def forward(self, x): return self.head(self.feat(x))

cnn = CNN().to(device)
opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)
for epoch in range(5):
    _, tr = run_epoch(cnn, ctrain_dl, opt)
    _, te = run_epoch(cnn, ctest_dl)
    print(f"epoch {epoch+1}: test acc {te:.3f}")
print(f"[CNN] CIFAR-10 test acc {te:.3f}   ← 같은 데이터, 합성곱을 얹었을 뿐")

### 관찰
- MLP는 CIFAR-10에서 약 0.45~0.50에 머문다 — 펴는 순간 위치 정보가 사라지기 때문.
- CNN은 같은 데이터로 5에폭이면 약 0.65~0.70. 합성곱이 **이웃한 픽셀의 무늬**를 보기 때문에 사진에 강하다.

## Lab 3 · 전이학습 — ResNet의 FC만 갈아 끼운다

처음부터 배우면 오래 걸린다. ImageNet에서 이미 배운 **ResNet18**을 가져와, 앞쪽(특징 추출)은 **얼리고(freeze)** 마지막 완전연결층(fc)만 우리 10개 클래스로 바꾼다. CIFAR-10을 224로 키우고 ImageNet 통계로 정규화해 맞춘다.

In [ ]:
from torchvision import models
from torchvision.models import ResNet18_Weights

# ImageNet 정규화 + 224 리사이즈 (사전학습 모델의 입력 규격에 맞춘다)
tf224 = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
# 빠른 체험을 위해 일부만 사용
from torch.utils.data import Subset
tr_sub = Subset(datasets.CIFAR10("data", train=True,  transform=tf224), range(4000))
te_sub = Subset(datasets.CIFAR10("data", train=False, transform=tf224), range(1000))
tr_dl = DataLoader(tr_sub, batch_size=64, shuffle=True)
te_dl = DataLoader(te_sub, batch_size=128)
print("전이학습용 subset:", len(tr_sub), "/", len(te_sub))

In [ ]:
net = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for p in net.parameters():         # 앞쪽 전부 얼림
    p.requires_grad = False
net.fc = nn.Linear(net.fc.in_features, 10)   # 마지막 층만 새로 (여기만 학습)
net = net.to(device)

# 학습되는 파라미터가 fc뿐임을 확인
trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
total = sum(p.numel() for p in net.parameters())
print(f"학습 파라미터 {trainable:,} / 전체 {total:,}  ({trainable/total*100:.2f}%)")

opt = torch.optim.Adam(net.fc.parameters(), lr=1e-3)   # fc만 최적화
for epoch in range(2):
    _, tr = run_epoch(net, tr_dl, opt)
    _, te = run_epoch(net, te_dl)
    print(f"epoch {epoch+1}: test acc {te:.3f}")

### 한 걸음 더 · 파인튜닝(fine-tuning)

fc만 바꿔 학습하면 앞쪽 층은 ImageNet 그대로다. 정확도를 더 높이려면 **앞쪽 일부를 조금 녹여(unfreeze)** 아주 낮은 학습률로 함께 미세 조정한다. 새 도메인이 ImageNet과 다를수록 효과가 크다.

In [ ]:
# 마지막 residual block(layer4)만 녹이고, 아주 낮은 lr로 함께 학습
for p in net.layer4.parameters():
    p.requires_grad = True

params = [p for p in net.parameters() if p.requires_grad]
print("파인튜닝 학습 파라미터:", sum(p.numel() for p in params), "개")

opt = torch.optim.Adam(params, lr=1e-4)   # 특징을 망치지 않게 lr을 낮춘다
for epoch in range(2):
    _, tr = run_epoch(net, tr_dl, opt)
    _, te = run_epoch(net, te_dl)
    print(f"[fine-tune] epoch {epoch+1}: test acc {te:.3f}")

### 관찰
- 학습하는 건 전체의 **1%도 안 되는 fc 층**뿐인데, 4,000장·2에폭으로 처음부터 배운 CNN을 금방 넘어선다.
- 앞쪽 층이 ImageNet에서 배운 '엣지→텍스처→개체' 특징을 그대로 빌려 쓰기 때문. **소량 데이터일수록 전이학습이 실무의 기본**이다.

## Lab 4 · CPU vs GPU — 같은 학습, 다른 시간

GPU는 같은 곱셈·덧셈을 수천 개 동시에 한다. 같은 CNN 학습 스텝을 CPU와 GPU에서 각각 돌려 시간을 재 본다.

In [ ]:
import time

def bench(dev, steps=30):
    m = CNN().to(dev)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    it = iter(ctrain_dl)
    xb, yb = next(it); xb, yb = xb.to(dev), yb.to(dev)
    if dev == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(steps):
        opt.zero_grad()
        loss = loss_fn(m(xb), yb)
        loss.backward(); opt.step()
    if dev == "cuda": torch.cuda.synchronize()
    return time.time() - t0

cpu_t = bench("cpu")
print(f"CPU: {cpu_t:.2f}s / 30 스텝")
if torch.cuda.is_available():
    gpu_t = bench("cuda")
    print(f"GPU: {gpu_t:.2f}s / 30 스텝  →  약 {cpu_t/gpu_t:.1f}배 빠름")
else:
    print("GPU 런타임이 아니다 — 런타임 유형을 T4로 바꾸면 비교가 나온다")

### 관찰
- 작은 CNN이라도 GPU가 수 배~수십 배 빠르다. 층이 깊고 이미지가 클수록 격차는 더 벌어진다.
- 다만 **GPU가 늘 답은 아니다** — 데이터가 작거나 모델이 가벼우면, GPU로 옮기는 오버헤드가 더 클 수 있다.

## Lab 5 · VLM — 이미지를 말로 묻는다 (NVIDIA API)

전통 CV는 라벨을 미리 정해야 했다. **VLM(비전 언어 모델)은 이미지에 자연어로 묻는다.** NVIDIA API의 비전 모델도 OpenAI 호환 `/v1` 규격이라, 이미지를 base64로 실어 보내면 된다.

CIFAR 테스트 이미지 한 장을 저장해서, 우리 CNN의 예측과 VLM의 답을 비교해 본다.

In [ ]:
%pip install -q openai

In [ ]:
import getpass, os
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...): ")

In [ ]:
# CIFAR 테스트 이미지 한 장을 크게 저장 (VLM에 보내기 좋게)
from torchvision import datasets
import torchvision.transforms.functional as F
raw = datasets.CIFAR10("data", train=False, download=True)
pil_img, label = raw[7]                 # 아무 인덱스
pil_big = pil_img.resize((224, 224))
pil_big.save("sample.jpg")
print("정답 라벨:", raw.classes[label])
pil_big

In [ ]:
import base64
from openai import OpenAI
client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)

img_b64 = base64.b64encode(open("sample.jpg","rb").read()).decode()
r = client.chat.completions.create(
    model="meta/llama-3.2-11b-vision-instruct",
    messages=[{"role":"user","content":[
        {"type":"text","text":"이 사진에 무엇이 있나? 한 문장 한국어로. 그리고 다음 중 무엇에 가장 가까운지 하나만: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck."},
        {"type":"image_url","image_url":{"url":f"data:image/jpeg;base64,{img_b64}"}}]}],
    max_tokens=120,
)
print(r.choices[0].message.content)

### 관찰
- VLM은 우리가 학습시키지 않았는데도 사진을 설명하고 분류한다 — **라벨을 미리 정하지 않아도** 되는 게 핵심 차이.
- 9~10강의 에이전트 루프에 이 호출을 도구로 붙이면, 에이전트가 **텍스트뿐 아니라 이미지도 읽는 눈**을 갖는다.
- 다만 정밀 좌표·실시간 대량 검출은 아직 전용 CV(YOLO 등)가 낫다. 유연함이 필요하면 VLM, 정밀·속도가 필요하면 전용 모델 — 섞어 쓴다.

## 체크리스트

**딥러닝 기초**
- [ ] MLP로 MNIST를 학습시키고 테스트 정확도를 확인했다
- [ ] `zero_grad → backward → step` 세 줄이 학습 루프의 핵심임을 안다

**CNN**
- [ ] CIFAR-10에서 MLP의 한계(펴면 이웃이 사라짐)를 정확도로 확인했다
- [ ] 같은 데이터에 CNN을 얹어 정확도가 오르는 것을 봤다

**전이학습**
- [ ] ResNet18의 fc만 바꾸고 앞쪽을 얼려 소량 데이터로 학습했다
- [ ] 학습 파라미터가 전체의 1% 미만임을 확인했다

**GPU**
- [ ] 같은 학습을 CPU/GPU에서 돌려 속도 차를 체감했다
- [ ] GPU가 늘 답은 아닌 경우를 안다

**VLM**
- [ ] NVIDIA API 비전 모델에 이미지+질문을 보내 답을 받았다
- [ ] VLM과 전통 CV를 언제 각각 쓰는지 안다